#

In [10]:
import os
from pathlib import Path
import requests
import json
import pandas as pd

In [11]:
OPENALEX_API_KEY = os.environ.get("OPENALEX_API_KEY")
DATA = Path("data")
os.makedirs(DATA, exist_ok=True)

In [15]:
def decode_abstract(index):
    words = sorted((pos, w) for w, positions in index.items() for pos in positions)
    return " ".join(w for _, w in words)

In [ ]:
RESEARCH_QUESTION = "Will heavy rainfall in the Alps become more frequent as the climate warms?"

INCLUSION_CRITERION = "The study makes projections for the future (not only an analysis of past data)."

cache = DATA / "openalex.json"

params_oa = {"search": "extreme precipitation Alps climate change",
                "filter": "has_abstract:true,type:article,publication_year:>2014",
                "per_page": 25,
                "select": "id,doi,title,publication_year,abstract_inverted_index"}
if OPENALEX_API_KEY:
    params_oa["api_key"] = OPENALEX_API_KEY


if not cache.exists():
    r = requests.get("https://api.openalex.org/works", params=params_oa, timeout=60)
    r.raise_for_status()
    records = [{"id": w["id"], "doi": w["doi"], "year": w["publication_year"], "title": w["title"],
                "abstract": decode_abstract(w["abstract_inverted_index"])}
               for w in r.json()["results"] if w.get("abstract_inverted_index")]
    cache.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")



papers = pd.DataFrame([{"id": w["id"], "year": w["year"], "title": w["title"],
                        "abstract": w["abstract"]}
                       for w in json.loads(cache.read_text())
                       if w.get("abstract")])
print(len(papers), "papers")
papers.head()

25 papers


,id,year,title,abstract
0,https://openalex.org/W2058420228,2015,Heavy precipitation in a changing climate: Doe...,Climate models project that heavy precipitatio...
1,https://openalex.org/W2793629918,2018,Changing seasonality of moderate and extreme p...,The intensity of precipitation events is expec...
2,https://openalex.org/W2466628033,2016,Decreasing spatial dependence in extreme snowf...,Whereas changes in magnitude of geophysical ex...
3,https://openalex.org/W2999656131,2020,Contrasting seasonal changes in total and inte...,Changes in precipitation over the European Alp...
4,https://openalex.org/W2756253087,2017,Local-scale changes in mean and heavy precipit...,High-resolution climate information provided b...
